# Lesson 2.1: Prompting Fundamentals

**Duration:** 45 minutes  
**Level:** Beginner  

---

## Learning Objectives

By the end of this lesson, you will:
- Understand prompt anatomy (system, user, assistant messages)
- Master zero-shot and few-shot prompting techniques
- Learn the "Perfect Prompt" template framework
- Implement structured outputs (JSON, markdown)
- Apply common prompt design patterns

---

## The Prompt Engineering Mindset

> "The difference between a $100K AI project and a failed POC is often just the prompt."

Effective prompting is about **clear communication**. Think of it as writing precise specifications for a highly capable but literal-minded assistant.

## Setup

In [ ]:
!pip install google-generativeai langchain-google-genai python-dotenv -q

In [ ]:
import os
os.environ['GOOGLE_API_KEY'] = 'key here'

In [ ]:
import json
from typing import List, Dict
from google import genai
from google.genai import types

client = genai.Client()

# Initialize model
response = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=f"Hello",)

print(response)
print("Setup complete!")

---

## 1. Anatomy of a Prompt

### Message Types in LLM Conversations

| Message Type | Purpose | Example |
|--------------|---------|----------|
| **System** | Sets behavior, persona, rules | "You are a customer service agent for TechCorp..." |
| **User** | The actual request/question | "How do I reset my password?" |
| **Assistant** | Model's response (or examples) | "To reset your password, follow these steps..." |

### Industry Example: E-commerce Customer Service Bot

In [ ]:
# Using Gemini's chat interface with system instruction

system_instruction = """
You are a customer service representative for ShopMax, an online electronics retailer.

Your responsibilities:
- Help customers with order inquiries, returns, and product questions
- Be friendly, professional, and concise
- For refund requests, inform customers about our 30-day return policy
- If you don't know something, offer to connect them with a specialist

Company policies:
- 30-day return window for unopened items
- 15-day return window for opened items
- Free shipping on orders over $50
"""


# Simulate customer interactions
customer_queries = [
    "Hi, I ordered a laptop 3 weeks ago and I want to return it. It's still sealed.",
    "What's your free shipping threshold?",
    "The TV I bought has a dead pixel. What can I do?"
]

print("CUSTOMER SERVICE BOT DEMO")
print("="*60)

for query in customer_queries:
    print(f"\nCustomer: {query}")
    response = client.models.generate_content(
        model="gemini-3-flash-preview",
        config=types.GenerateContentConfig(
            system_instruction=system_instruction),
        contents=query
    )
    print(f"\nAgent: {response.text}")
    print("-"*60)

In [ ]:
# LangChain approach with message types
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.3)

# Demonstrating conversation with message types
messages = [
    SystemMessage(content="You are a helpful financial advisor assistant. Provide clear, actionable advice."),
    HumanMessage(content="I have $10,000 to invest. I'm 30 years old and have moderate risk tolerance."),
]

print("LANGCHAIN MESSAGE TYPES DEMO")
print("="*60)

response = llm.invoke(messages)
print(f"\nFinancial Advice:")
print(response.content)

In [ ]:
response

---

## 2. Zero-Shot Prompting

### What is Zero-Shot?

Zero-shot prompting means asking the model to perform a task **without providing examples**. It relies on the model's pre-training knowledge.

**When to use:**
- Simple, well-defined tasks
- Common operations (translation, summarization)
- When you want quick results

**Limitations:**
- May not follow specific formats
- Less reliable for domain-specific tasks
- Inconsistent output structure

### Industry Example: Resume Screening System

In [ ]:
def zero_shot_classify(text: str, categories: List[str]) -> str:
    """
    Zero-shot classification - no examples provided.
    """
    categories_str = ", ".join(categories)
    
    prompt = f"""Classify the following text into one of these categories: {categories_str}

Text: {text}

Category:"""

    response = llm.invoke(prompt)
    return response.content.strip()

# Resume skills classification
resumes = [
    "5 years experience in Python, Django, PostgreSQL. Led team of 4 developers. Built scalable microservices.",
    "MBA from Harvard. 10 years in product management at FAANG. Launched 3 products with $50M+ revenue.",
    "Graphic designer with expertise in Figma, Adobe Creative Suite. Created brand identities for Fortune 500.",
    "Data scientist skilled in ML, TensorFlow, and statistical analysis. Published 5 papers on NLP.",
]

job_categories = ["Software Engineering", "Product Management", "Design", "Data Science", "Marketing"]

print("ZERO-SHOT RESUME CLASSIFICATION")
print("="*60)

for resume in resumes:
    category = zero_shot_classify(resume, job_categories)
    print(f"\nResume: {resume[:60]}...")
    print(f"   Category: {category}")

In [ ]:
# Zero-shot sentiment analysis for product reviews
reviews = [
    "This laptop is incredible! Fast, lightweight, and the battery lasts all day.",
    "Disappointed. The screen quality is mediocre and it runs hot under load.",
    "It's okay for the price. Nothing special but gets the job done.",
    "Best purchase I've made this year! Worth every penny.",
    "Good build quality but the software is buggy. Mixed feelings overall."
]

print("ZERO-SHOT PRODUCT REVIEW ANALYSIS")
print("="*60)

for review in reviews:
    sentiment = zero_shot_classify(review, ["positive", "negative", "neutral", "mixed"])
    print(f"\nReview: \"{review[:50]}...\"")
    print(f"   Sentiment: {sentiment}")

---

## 3. Few-Shot Prompting

### What is Few-Shot?

Few-shot prompting provides **examples** that demonstrate the desired behavior. The model learns the pattern from examples.

**Best Practices:**
- Use 3-5 diverse examples
- Keep format consistent across examples
- Include edge cases in examples
- Order can matter (put strongest examples last)

### Industry Example: Lead Qualification System

In [19]:
def few_shot_classify(text: str, examples: List[Dict[str, str]], categories: List[str]) -> str:
    """
    Few-shot classification with examples.
    """
    categories_str = ", ".join(categories)
    
    # Build examples section
    examples_str = "\n\n".join([
        f"Lead: {ex['text']}\nQualification: {ex['category']}"
        for ex in examples
    ])
    
    prompt = f"""Qualify sales leads into one of these categories: {categories_str}

{examples_str}

Lead: {text}
Qualification:"""
    
    response = llm.invoke(prompt)
    return response.content.strip()

# Lead qualification examples (the "shots")
lead_examples = [
    {
        "text": "CTO of a 500-person company, budget approved, needs solution this quarter.",
        "category": "Hot Lead"
    },
    {
        "text": "Marketing manager exploring options, no timeline, still evaluating vendors.",
        "category": "Warm Lead"
    },
    {
        "text": "Student researching for a class project, no budget or authority.",
        "category": "Cold Lead"
    },
    {
        "text": "VP of Engineering, active RFP process, decision next month.",
        "category": "Hot Lead"
    }
]

categories = ["Hot Lead", "Warm Lead", "Cold Lead"]

# New leads to qualify
new_leads = [
    "CEO of a startup, just raised Series A, looking to implement ASAP.",
    "Intern collecting information for their manager, no decision-making power.",
    "Director of Operations, interested but budget is Q3 next year.",
    "Founder with 10 employees, urgent need, budget constraints."
]

print("FEW-SHOT LEAD QUALIFICATION")
print("="*60)

for lead in new_leads:
    qualification = few_shot_classify(lead, lead_examples, categories)
    print(f"\nLead: {lead}")
    print(f"   Qualification: {qualification}")

FEW-SHOT LEAD QUALIFICATION

Lead: CEO of a startup, just raised Series A, looking to implement ASAP.
   Qualification: Hot Lead

Lead: Intern collecting information for their manager, no decision-making power.
   Qualification: Cold Lead

Lead: Director of Operations, interested but budget is Q3 next year.
   Qualification: Warm Lead

Lead: Founder with 10 employees, urgent need, budget constraints.
   Qualification: Hot Lead


In [20]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

# Define examples
examples = [
    {"input": "How do I reset my password?", "output": "account_support"},
    {"input": "When will my order arrive?", "output": "shipping_inquiry"},
    {"input": "This product is defective!", "output": "complaint"},
    {"input": "Can I get a bulk discount?", "output": "sales_inquiry"},
]

# Create example prompt template
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "Category: {output}")
])

# Create few-shot prompt
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples
)

# Final prompt
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "Classify customer inquiries into categories: account_support, shipping_inquiry, complaint, sales_inquiry, general_question"),
    few_shot_prompt,
    ("human", "{input}")
])

# Create chain
chain = final_prompt | llm

# Test
test_inputs = [
    "I want to change my email address",
    "Where's my package? It's been 10 days!",
    "Do you offer enterprise pricing?"
]

print("LANGCHAIN FEW-SHOT CLASSIFICATION")
print("="*60)

for inp in test_inputs:
    result = chain.invoke({"input": inp})
    print(f"\nInput: {inp}")
    print(f"   {result.content}")

LANGCHAIN FEW-SHOT CLASSIFICATION

Input: I want to change my email address
   Category: account_support

Input: Where's my package? It's been 10 days!
   Category: shipping_inquiry

Input: Do you offer enterprise pricing?
   Category: sales_inquiry


In [25]:
final_prompt.invoke({"input": "I need help with my order"}).messages

[SystemMessage(content='Classify customer inquiries into categories: account_support, shipping_inquiry, complaint, sales_inquiry, general_question', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='How do I reset my password?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Category: account_support', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='When will my order arrive?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Category: shipping_inquiry', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='This product is defective!', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Category: complaint', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Can I get a bulk discount?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Category:

---

## 4. The Perfect Prompt Template

### The 7-Component Framework

```
1. ROLE        - Who is the AI?
2. CONTEXT     - Background information
3. TASK        - Specific action to perform
4. INPUT       - What will be provided
5. OUTPUT      - Expected format/structure
6. CONSTRAINTS - Rules and limitations
7. EXAMPLES    - Sample inputs/outputs (optional)
```

### Industry Example: Automated Code Review System

In [26]:
class PerfectPromptTemplate:
    """
    The Perfect Prompt Template builder.
    """
    
    def __init__(
        self,
        role: str,
        context: str,
        task: str,
        input_format: str,
        output_format: str,
        constraints: List[str] = None,
        examples: List[Dict] = None
    ):
        self.role = role
        self.context = context
        self.task = task
        self.input_format = input_format
        self.output_format = output_format
        self.constraints = constraints or []
        self.examples = examples or []
    
    def build(self) -> str:
        """Build the complete prompt template."""
        parts = [
            f"# Role\nYou are {self.role}.",
            f"\n# Context\n{self.context}",
            f"\n# Task\n{self.task}",
            f"\n# Input Format\n{self.input_format}",
            f"\n# Output Format\n{self.output_format}",
        ]
        
        if self.constraints:
            constraints_str = "\n".join([f"- {c}" for c in self.constraints])
            parts.append(f"\n# Constraints\n{constraints_str}")
        
        if self.examples:
            examples_str = "\n\n".join([
                f"Input: {ex['input']}\nOutput: {ex['output']}"
                for ex in self.examples
            ])
            parts.append(f"\n# Examples\n{examples_str}")
        
        return "\n".join(parts)

# Code Review System
code_review_template = PerfectPromptTemplate(
    role="a senior software engineer with 10+ years of experience in code review",
    context="You are conducting code reviews for a fintech startup that prioritizes security, maintainability, and performance.",
    task="Review the provided code snippet and identify issues, suggest improvements, and highlight good practices.",
    input_format="A code snippet in Python, JavaScript, or another common language.",
    output_format="""Provide your review in this structure:
## Summary
[1-2 sentence overview]

## Issues Found
1. [Issue with severity: Critical/Major/Minor]
   - Description
   - Suggested fix

## Good Practices
- [What the code does well]

## Recommendations
- [Actionable improvements]""",
    constraints=[
        "Prioritize security vulnerabilities",
        "Be constructive, not critical",
        "Include code examples for suggested fixes",
        "Consider performance implications"
    ]
)

print("PERFECT PROMPT TEMPLATE: Code Review System")
print("="*60)
print(code_review_template.build())

PERFECT PROMPT TEMPLATE: Code Review System
# Role
You are a senior software engineer with 10+ years of experience in code review.

# Context
You are conducting code reviews for a fintech startup that prioritizes security, maintainability, and performance.

# Task
Review the provided code snippet and identify issues, suggest improvements, and highlight good practices.

# Input Format
A code snippet in Python, JavaScript, or another common language.

# Output Format
Provide your review in this structure:
## Summary
[1-2 sentence overview]

## Issues Found
1. [Issue with severity: Critical/Major/Minor]
   - Description
   - Suggested fix

## Good Practices
- [What the code does well]

## Recommendations
- [Actionable improvements]

# Constraints
- Prioritize security vulnerabilities
- Be constructive, not critical
- Include code examples for suggested fixes
- Consider performance implications


In [27]:
# Use the template for actual code review
code_to_review = '''
def get_user_data(user_id):
    query = f"SELECT * FROM users WHERE id = {user_id}"
    result = db.execute(query)
    return result

def authenticate(username, password):
    user = get_user_data(username)
    if user["password"] == password:
        return True
    return False
'''

from langchain.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage(content=code_review_template.build()),
    HumanMessage(content=f"Review this code:\n```python{code_to_review}```")
]

response = llm.invoke(messages)

print("CODE REVIEW RESULT")
print("="*60)
print(response.text)

CODE REVIEW RESULT
## Summary
The provided code snippet demonstrates basic user data retrieval and authentication logic but contains critical security vulnerabilities, particularly SQL injection and insecure password handling, along with a significant logical error in how authentication is performed.

## Issues Found

1.  **SQL Injection (Critical)**
    *   **Description:** The `get_user_data` function constructs SQL queries using f-strings with user-provided input (`user_id`). This is a classic SQL injection vulnerability. A malicious user could inject arbitrary SQL commands into the `user_id` parameter, potentially leading to data breaches, unauthorized access, or data manipulation.
    *   **Suggested fix:** Always use parameterized queries (prepared statements) provided by your database driver or ORM. This separates the SQL logic from the data, preventing injection attacks.
    *   **Code Example:**
        ```python
        # Assuming `db.execute` supports parameterized queries (

---

## 5. Structured Outputs

### Why Structured Output Matters

In production systems, you need **predictable, parseable outputs**:
- JSON for APIs and data pipelines
- Markdown for documentation
- Specific formats for downstream processing

### Industry Example: Invoice Data Extraction

In [28]:
# JSON Output - Invoice Extraction
invoice_text = """
INVOICE #INV-2024-0892
Date: January 15, 2024

From: TechSupply Inc.
123 Business Ave, San Francisco, CA 94102

To: Acme Corporation
456 Enterprise Blvd, New York, NY 10001

Items:
- MacBook Pro 16" (x3) - $2,499.00 each
- Dell Monitor 27" (x5) - $449.00 each
- Logitech Keyboard (x10) - $99.00 each

Subtotal: $10,732.00
Tax (8.5%): $912.22
Total: $11,644.22

Payment Due: February 15, 2024
"""

extraction_prompt = """
Extract structured data from this invoice. Return ONLY valid JSON.

Required JSON structure:
{
    "invoice_number": "string",
    "date": "YYYY-MM-DD",
    "vendor": {
        "name": "string",
        "address": "string"
    },
    "customer": {
        "name": "string",
        "address": "string"
    },
    "items": [
        {
            "description": "string",
            "quantity": "number",
            "unit_price": "number",
            "total": "number"
        }
    ],
    "subtotal": "number",
    "tax": "number",
    "total": "number",
    "due_date": "YYYY-MM-DD"
}

Invoice:
""" + invoice_text

response = llm.invoke(extraction_prompt, temperature=0, response_mime_type="application/json")

print("INVOICE DATA EXTRACTION")
print("="*60)

try:
    extracted_data = json.loads(response.content)
    print(json.dumps(extracted_data, indent=2))
except json.JSONDecodeError:
    print("Raw response:")
    print(response.content)

INVOICE DATA EXTRACTION
{
  "invoice_number": "INV-2024-0892",
  "date": "2024-01-15",
  "vendor": {
    "name": "TechSupply Inc.",
    "address": "123 Business Ave, San Francisco, CA 94102"
  },
  "customer": {
    "name": "Acme Corporation",
    "address": "456 Enterprise Blvd, New York, NY 10001"
  },
  "items": [
    {
      "description": "MacBook Pro 16\"",
      "quantity": 3,
      "unit_price": 2499.0,
      "total": 7497.0
    },
    {
      "description": "Dell Monitor 27\"",
      "quantity": 5,
      "unit_price": 449.0,
      "total": 2245.0
    },
    {
      "description": "Logitech Keyboard",
      "quantity": 10,
      "unit_price": 99.0,
      "total": 990.0
    }
  ],
  "subtotal": 10732.0,
  "tax": 912.22,
  "total": 11644.22,
  "due_date": "2024-02-15"
}


---

## 6. Prompt Design Patterns

### Common Patterns Used in Production

| Pattern | Description | Use Case |
|---------|-------------|----------|
| Role-Based | Assign a persona | Expert advice, domain tasks |
| Constraint-Based | Set explicit rules | Safe outputs, specific formats |
| Step-by-Step | Break down complex tasks | Analysis, problem solving |
| Output Anchoring | Start the response | Ensure format compliance |

In [29]:
class PromptPatterns:
    """Common prompt design patterns."""
    
    @staticmethod
    def role_based(role: str, task: str) -> str:
        """Role-based prompting - assign expertise."""
        return f"""You are {role}.
                {task}"""
    
    @staticmethod
    def constraint_based(task: str, constraints: List[str]) -> str:
        """Constraint-based - explicit rules."""
        constraints_str = "\n".join([f"- {c}" for c in constraints])
        return f"""{task}
            IMPORTANT RULES:
            {constraints_str}"""
    
    @staticmethod
    def step_by_step(task: str, steps: List[str]) -> str:
        """Step-by-step - structured approach."""
        steps_str = "\n".join([f"{i+1}. {s}" for i, s in enumerate(steps)])
        return f"""{task}

            Follow these steps:
            {steps_str}

            Show your work for each step."""
    
    @staticmethod
    def output_anchoring(task: str, anchor: str) -> str:
        """Output anchoring - start the response format."""
        return f"""{task}

            Begin your response with:
            {anchor}"""

patterns = PromptPatterns()

# Demo each pattern
print("PROMPT DESIGN PATTERNS")
print("="*60)

# Pattern 1: Role-Based
print("\nROLE-BASED PATTERN")
print("-"*40)
prompt = patterns.role_based(
    "a senior tax accountant with 15 years of experience",
    "Explain the tax implications of converting a traditional IRA to a Roth IRA."
)
response = llm.invoke(prompt)
print(f"Response: {response.content[:300]}...")

PROMPT DESIGN PATTERNS

ROLE-BASED PATTERN
----------------------------------------
Response: Alright, let's break down the tax implications of converting a Traditional IRA to a Roth IRA. As a senior tax accountant, I've guided many clients through this decision, and it's a strategic move with significant, immediate, and long-term tax consequences.

**The Core Principle: It's Generally a Tax...


In [30]:
# Pattern 2: Constraint-Based
print("\nCONSTRAINT-BASED PATTERN")
print("-"*40)
prompt = patterns.constraint_based(
    "Write a product description for a children's educational toy.",
    [
        "Keep language appropriate for all ages",
        "Do not make exaggerated claims",
        "Maximum 100 words",
        "Include safety information",
        "Avoid superlatives like 'best' or 'greatest'"
    ]
)
response = llm.invoke(prompt)
print(f"Response:\n{response.content}")


CONSTRAINT-BASED PATTERN
----------------------------------------
Response:
**Rainbow Stacking Shapes**

Discover a world of color and form with Rainbow Stacking Shapes! Little hands can stack the smooth, vibrant pieces, sort them by shape, and match them to the sturdy wooden base. This engaging toy supports the development of fine motor skills, shape recognition, and early problem-solving. Crafted from durable, non-toxic wood with child-safe paints. Recommended for ages 18 months and up. Adult supervision is advised during playtime.


In [31]:
# Pattern 3: Step-by-Step
print("\nSTEP-BY-STEP PATTERN")
print("-"*40)
prompt = patterns.step_by_step(
    "Analyze this business problem: A SaaS company has 30% monthly churn rate.",
    [
        "Identify potential root causes",
        "Prioritize causes by likelihood and impact",
        "Suggest 3 specific interventions",
        "Estimate potential impact of each intervention"
    ]
)
response = llm.invoke(prompt)
print(f"Response:\n{response.content}")


STEP-BY-STEP PATTERN
----------------------------------------
Response:
A 30% monthly churn rate for a SaaS company is an extremely critical situation. To put it in perspective, a healthy SaaS company typically aims for a monthly churn rate of 2-5% for SMBs and less than 1% for enterprise customers. A 30% rate means that nearly a third of the customer base is leaving every single month, making sustainable growth virtually impossible and indicating fundamental problems within the business.

---

### 1. Identify Potential Root Causes

Given the severity of a 30% monthly churn rate, it's highly likely that multiple systemic issues are at play.

**A. Product-Related Issues:**
*   **Poor Product-Market Fit:** The product doesn't genuinely solve a critical problem for its target audience, or the value proposition is unclear/weak.
*   **Ineffective Onboarding:** Users fail to understand how to use the product, achieve their "aha!" moment, or integrate it into their workflow quickly. They get

---

## 7. Practical Exercise: Build a Customer Feedback Analyzer

Let's combine all the concepts to build a production-ready feedback analysis system.

In [32]:
class FeedbackAnalyzer:
    """
    Production-ready customer feedback analyzer.
    Demonstrates: Templates, structured output, few-shot learning.
    """
    
    def __init__(self):
        self.system_instruction = """
            You are a customer feedback analyst for a B2B software company.

            Your role:
            - Analyze customer feedback to extract actionable insights
            - Identify sentiment, urgency, and key themes
            - Route feedback to appropriate teams
            - Flag critical issues that need immediate attention

            Always provide structured, actionable analysis.
            """

        self.llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.5)
    
    def analyze(self, feedback: str) -> dict:
        """Analyze customer feedback and return structured insights."""
        
        prompt = f'''
            Analyze this customer feedback and return JSON:

            FEEDBACK:
            "{feedback}"

            Return this exact JSON structure:
            {{
                "sentiment": "positive or negative or neutral or mixed",
                "urgency": "critical or high or medium or low",
                "category": "bug_report or feature_request or praise or complaint or question",
                "key_themes": ["theme1", "theme2"],
                "route_to": "engineering or product or support or sales or leadership",
                "summary": "One sentence summary",
                "suggested_action": "Recommended next step"
            }}
            '''
        
        response = self.llm.invoke(
            [
                SystemMessage(content=self.system_instruction),
                HumanMessage(content=prompt)
            ],
            temperature=0,
            response_mime_type="application/json"
        )

        print(response.content)  # Debug: print raw response
        
        return json.loads(response.content)
    
    def generate_response(self, feedback: str, analysis: dict) -> str:
        """Generate an appropriate response based on analysis."""
        
        prompt = f'''
            Based on this customer feedback and analysis, draft a response.

            FEEDBACK: "{feedback}"

            ANALYSIS:
            - Sentiment: {analysis['sentiment']}
            - Category: {analysis['category']}
            - Urgency: {analysis['urgency']}

            Write a professional, empathetic response that:
            1. Acknowledges their feedback
            2. Addresses their concern appropriately
            3. Provides a clear next step

            Keep it under 100 words.
            '''
        response = self.llm.invoke(
            [
                SystemMessage(content=self.system_instruction),
                HumanMessage(content=prompt)
            ],
        )
        
        return response.text

# Demo the analyzer
analyzer = FeedbackAnalyzer()

feedbacks = [
    "Your product is amazing! The quality exceeded my expectations.",
    "The app keeps crashing every time I try to upload a photo.",
    "It would be great if you could add dark mode to the mobile app.",
]

print("CUSTOMER FEEDBACK ANALYZER")
print("="*70)

for feedback in feedbacks:
    print(f"\nFeedback: \"{feedback}\"")
    print("-"*60)
    
    analysis = analyzer.analyze(feedback)
    
    print(f"Analysis:")
    print(f"   Sentiment: {analysis['sentiment']} | Urgency: {analysis['urgency']}")
    print(f"   Category: {analysis['category']} | Route to: {analysis['route_to']}")
    print(f"   Themes: {', '.join(analysis['key_themes'])}")
    print(f"   Action: {analysis['suggested_action']}")
    
    response = analyzer.generate_response(feedback, analysis)
    print(f"Response: {response}")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


CUSTOMER FEEDBACK ANALYZER

Feedback: "Your product is amazing! The quality exceeded my expectations."
------------------------------------------------------------
{
    "sentiment": "positive",
    "urgency": "low",
    "category": "praise",
    "key_themes": [
        "product quality",
        "customer satisfaction"
    ],
    "route_to": "product",
    "summary": "The customer is highly satisfied with the product, specifically praising its quality.",
    "suggested_action": "Acknowledge the positive feedback and share with the product team to boost morale and potentially use for testimonials."
}
Analysis:
   Sentiment: positive | Urgency: low
   Category: praise | Route to: product
   Themes: product quality, customer satisfaction
   Action: Acknowledge the positive feedback and share with the product team to boost morale and potentially use for testimonials.
Response: Thank you for sharing such wonderful feedback! We're thrilled to hear our product has exceeded your expectations 

---

## Key Takeaways

### 1. Prompt Structure Matters
- Use system messages for behavior/persona
- User messages for requests
- Clear separation improves consistency

### 2. Zero-Shot vs Few-Shot
- Zero-shot: Simple tasks, common operations
- Few-shot: Domain-specific, format-sensitive tasks
- 3-5 diverse examples is usually optimal

### 3. The Perfect Prompt Template
- Role, Context, Task, Input, Output, Constraints, Examples
- More structure = more consistent results

### 4. Structured Outputs
- Use JSON for APIs and data pipelines
- Specify exact schema in prompt
- Use response_mime_type when available

### 5. Design Patterns
- Role-based: Expertise injection
- Constraint-based: Safety and format control
- Step-by-step: Complex reasoning
- Output anchoring: Format compliance

---

## Next Steps

Proceed to **Lesson 2.2: Advanced Prompting Techniques** to learn:
- Chain-of-Thought (CoT) prompting
- Tree-of-Thought (ToT)
- ReAct pattern
- Self-consistency